# 03. Experimentación y Selección de Modelos

Con los datos limpios, enriquecidos y escalados, es hora de encontrar el algoritmo predictivo ganador.

### Instrucciones Generales:
1. **Validación:** No entrenes y midas sobre el mismo conjunto (sobreajuste). Recuerda haber dividido en Entrenamiento y Prueba antes.
2. **Entrenamiento Base:** Entrena los siguientes modelos base con tu set de Entrenamiento y compáralos usando RMSE (Error Cuadrático Medio):
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation (Validación Cruzada):** Para tener una métrica robusta, usa `cross_val_score` en el set de Entrenamiento para cada uno de los modelos anteriores.
4. **Ajuste Fino (Fine Tuning):** Toma el modelo ganador y busca sus mejores hiperparámetros. Utiliza un `GridSearchCV` explorando el número de estimadores (`n_estimators`), las características máximas (`max_features`), etc.
5. **Conclusión y Benchmark (IMPORTANTE):** Redacta una conclusión comparando los algoritmos. Explica por qué escogiste el modelo final y valida tu decisión calculando el RMSE sobre tu Set de Prueba que habías reservado. Documenta si alguno de tus modelos se sobreajusto o subajusto. Recuerda que el modelo final no puede tener esos problemas!


In [26]:
import warnings
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

**CARGA DATASET**

Como este notebook debe ser autosuficiente, es mejor rehacer el preprocesamiento del dataset dentro de él.

In [27]:
train = pd.read_csv("../data/interim/train.csv")
test = pd.read_csv("../data/interim/test.csv")

median_bedrooms = train["total_bedrooms"].median()
train["total_bedrooms"] = train["total_bedrooms"].fillna(median_bedrooms)
test["total_bedrooms"] = test["total_bedrooms"].fillna(median_bedrooms)

In [28]:
def add_features_base(df):
    df = df.copy()
    df["rooms_per_household"] = df["total_rooms"] / df["households"]
    df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
    df["population_per_household"] = df["population"] / df["households"]
    return df

def add_features_extra(df):
    df = df.copy()
    df["bedrooms_per_household"] = df["total_bedrooms"] / df["households"]
    df["rooms_per_person"] = df["total_rooms"] / df["population"]
    df["income_x_rooms_per_household"] = df["median_income"] * df["rooms_per_household"]
    return df

**CONSTRUCCION DE SETS A,B & C** 

Con los datos limpios y preparados, esta fase tiene como objetivo identificar el algoritmo y el conjunto de variables con mejor capacidad predictiva.
Se comparan cuatro modelos base (`LinearRegression`, `SGDRegressor`, `DecisionTreeRegressor` y `RandomForestRegressor`) sobre tres configuraciones de variables:

- **Set A:** variables originales
- **Set B:** variables originales + features base
- **Set C:** variables originales + features base + features ampliadas

La selección del modelo final se basa en validación cruzada, priorizando RMSE como métrica principal y complementando el análisis con MAE y R². El criterio de decisión no se limita a maximizar precisión, sino a encontrar el mejor balance entre desempeño y generalización.

In [29]:
# MODELO A: originales
train_A = train.copy()
test_A = test.copy()

# MODELO B: originales + 3 base
train_B = add_features_base(train)
test_B = add_features_base(test)

# MODELO C: originales + 3 base + 3 extra
train_C = add_features_base(train)
train_C = add_features_extra(train_C)

test_C = add_features_base(test)
test_C = add_features_extra(test_C)

**ONE-HOT ENCODING y SPLIT X & Y**

In [30]:
def encode_data(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    if "ocean_proximity" in train_df.columns:
        train_df = pd.get_dummies(train_df, columns=["ocean_proximity"], drop_first=True)
        test_df = pd.get_dummies(test_df, columns=["ocean_proximity"], drop_first=True)

    train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)
    return train_df, test_df

train_A, test_A = encode_data(train_A, test_A)
train_B, test_B = encode_data(train_B, test_B)
train_C, test_C = encode_data(train_C, test_C)

In [31]:
def split_xy(df):
    X = df.drop("median_house_value", axis=1)
    y = df["median_house_value"]
    return X, y

X_A, y_A = split_xy(train_A)
X_B, y_B = split_xy(train_B)
X_C, y_C = split_xy(train_C)

X_test_A, y_test_A = split_xy(test_A)
X_test_B, y_test_B = split_xy(test_B)
X_test_C, y_test_C = split_xy(test_C)

**LOG TRANSFORM y ESCALADO**

In [32]:
def add_log_income(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    X_train["median_income_log"] = np.log1p(X_train["median_income"])
    X_test["median_income_log"] = np.log1p(X_test["median_income"])

    return X_train, X_test

X_A, X_test_A = add_log_income(X_A, X_test_A)
X_B, X_test_B = add_log_income(X_B, X_test_B)
X_C, X_test_C = add_log_income(X_C, X_test_C)

In [33]:
def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

X_A_scaled, X_test_A_scaled, scaler_A = scale_data(X_A, X_test_A)
X_B_scaled, X_test_B_scaled, scaler_B = scale_data(X_B, X_test_B)
X_C_scaled, X_test_C_scaled, scaler_C = scale_data(X_C, X_test_C)

**EVALUACION CON VALIDACION CRUZADA**

In [34]:
def evaluate_cv(model, X, y, cv=5):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        try:
            scores_rmse = -cross_val_score(
                model, X, y,
                scoring="neg_root_mean_squared_error",
                cv=cv,
                error_score=np.nan
            )
            scores_mae = -cross_val_score(
                model, X, y,
                scoring="neg_mean_absolute_error",
                cv=cv,
                error_score=np.nan
            )
            scores_r2 = cross_val_score(
                model, X, y,
                scoring="r2",
                cv=cv,
                error_score=np.nan
            )

            return {
                "rmse_mean": np.nanmean(scores_rmse),
                "rmse_std": np.nanstd(scores_rmse),
                "mae_mean": np.nanmean(scores_mae),
                "r2_mean": np.nanmean(scores_r2)
            }

        except Exception:
            return {
                "rmse_mean": np.nan,
                "rmse_std": np.nan,
                "mae_mean": np.nan,
                "r2_mean": np.nan
            }

**MODELOS BASE CORREGIDO**

In [35]:
models_scaled = {
    "LinearRegression": LinearRegression(),
    "SGDRegressor": SGDRegressor(
        random_state=42,
        max_iter=5000,
        tol=1e-4,
        penalty="l2",
        alpha=0.0001,
        learning_rate="invscaling",
        eta0=0.01,
        early_stopping=True
    )
}

models_unscaled = {
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(
        random_state=42,
        n_estimators=200,
        n_jobs=-1
    )
}

**COMPARACION**

In [36]:
results = []

datasets = {
    "A_originales": (X_A, X_A_scaled, y_A),
    "B_base_features": (X_B, X_B_scaled, y_B),
    "C_features_ampliadas": (X_C, X_C_scaled, y_C)
}

for dataset_name, (X_raw, X_scaled, y) in datasets.items():
    for model_name, model in models_scaled.items():
        metrics = evaluate_cv(model, X_scaled, y)
        results.append({
            "dataset": dataset_name,
            "model": model_name,
            **metrics
        })

    for model_name, model in models_unscaled.items():
        metrics = evaluate_cv(model, X_raw, y)
        results.append({
            "dataset": dataset_name,
            "model": model_name,
            **metrics
        })

results_df = pd.DataFrame(results).sort_values(by="rmse_mean")
results_df

,dataset,model,rmse_mean,rmse_std,mae_mean,r2_mean
3,A_originales,RandomForest,4.928423e+04,1.507979e+02,3.218106e+04,8.171840e-01
11,C_features_ampliadas,RandomForest,4.970024e+04,6.461965e+02,3.260318e+04,8.140517e-01
7,B_base_features,RandomForest,4.996872e+04,5.993277e+02,3.278463e+04,8.120031e-01
8,C_features_ampliadas,LinearRegression,6.687651e+04,1.949434e+03,4.786999e+04,6.628391e-01
4,B_base_features,LinearRegression,6.794369e+04,3.737103e+02,4.931669e+04,6.525209e-01
0,A_originales,LinearRegression,6.832372e+04,3.549415e+02,4.970681e+04,6.486704e-01
2,A_originales,DecisionTree,6.842528e+04,8.389222e+02,4.389995e+04,6.475800e-01
10,C_features_ampliadas,DecisionTree,7.040578e+04,1.973750e+03,4.515431e+04,6.261362e-01
6,B_base_features,DecisionTree,7.077641e+04,2.117568e+03,4.569835e+04,6.220823e-01
5,B_base_features,SGDRegressor,2.762948e+08,1.885919e+08,8.638173e+07,-8.460672e+06


In [38]:
summary_table = results_df.copy()

summary_table["rmse_mean"] = summary_table["rmse_mean"].round(2)
summary_table["rmse_std"] = summary_table["rmse_std"].round(2)
summary_table["mae_mean"] = summary_table["mae_mean"].round(2)
summary_table["r2_mean"] = summary_table["r2_mean"].round(4)

summary_table = summary_table.rename(columns={
    "dataset": "Set de variables",
    "model": "Modelo",
    "rmse_mean": "RMSE promedio",
    "rmse_std": "Desv. RMSE",
    "mae_mean": "MAE promedio",
    "r2_mean": "R² promedio"
})

summary_table.style.background_gradient(subset=["RMSE promedio", "MAE promedio"], cmap="Blues_r") \
                   .background_gradient(subset=["R² promedio"], cmap="Greens") \
                   .set_caption("Benchmark de modelos con validación cruzada")

,Set de variables,Modelo,RMSE promedio,Desv. RMSE,MAE promedio,R² promedio
3,A_originales,RandomForest,49284.230000,150.800000,32181.060000,0.817200
11,C_features_ampliadas,RandomForest,49700.240000,646.200000,32603.180000,0.814100
7,B_base_features,RandomForest,49968.720000,599.330000,32784.630000,0.812000
8,C_features_ampliadas,LinearRegression,66876.510000,1949.430000,47869.990000,0.662800
4,B_base_features,LinearRegression,67943.690000,373.710000,49316.690000,0.652500
0,A_originales,LinearRegression,68323.720000,354.940000,49706.810000,0.648700
2,A_originales,DecisionTree,68425.280000,838.920000,43899.950000,0.647600
10,C_features_ampliadas,DecisionTree,70405.780000,1973.750000,45154.310000,0.626100
6,B_base_features,DecisionTree,70776.410000,2117.570000,45698.350000,0.622100
5,B_base_features,SGDRegressor,276294841.300000,188591876.740000,86381728.940000,-8460671.583200


**Interpretación de resultados**

La comparación mediante validación cruzada muestra que el mejor desempeño corresponde a **RandomForestRegressor con el set A (variables originales)**, con un RMSE promedio de aproximadamente **49,284** y un MAE promedio cercano a **32,181**.

En segundo lugar se ubica **RandomForest con el set C**, seguido por **RandomForest con el set B**. Esto indica que el algoritmo de ensamble es claramente superior a los modelos lineales y al árbol de decisión individual para este problema, ya que logra capturar relaciones no lineales entre ingreso, localización y características habitacionales.

Un hallazgo importante es que las variables derivadas no mejoraron el desempeño fuera de muestra respecto al conjunto original. Aunque los sets B y C incorporan información conceptualmente útil, su aporte no superó al set A en validación cruzada. En este caso, el feature engineering adicional incrementó la complejidad del modelo sin traducirse en una mejora efectiva de generalización.

`LinearRegression` mostró un desempeño intermedio y se mantuvo como baseline interpretable, mientras que `DecisionTreeRegressor` presentó un RMSE más alto y mayor sensibilidad al sobreajuste. Por su parte, `SGDRegressor` exhibió errores extremadamente altos e inestabilidad numérica, por lo que fue descartado como candidato viable.

**Selección del set y modelo ganador**

Con base en los resultados observados, se selecciona como solución final:

- **Set ganador:** `A_originales`
- **Modelo ganador:** `RandomForestRegressor`

La decisión se sustenta en que esta combinación obtuvo el menor RMSE promedio entre todas las alternativas evaluadas, así como el MAE más bajo dentro de los modelos competitivos. Además, representa la opción más parsimoniosa, ya que logra el mejor desempeño sin depender de un aumento innecesario de variables.

Desde una perspectiva metodológica, este resultado confirma que agregar más features no garantiza una mejora del modelo. La evidencia empírica favorece una solución más simple, robusta y con mejor capacidad de generalización.

**EVALUACION OVERFITTING**

In [39]:
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_A, y_A, test_size=0.2, random_state=42
)

rf_base = RandomForestRegressor(
    random_state=42,
    n_estimators=200,
    n_jobs=-1
)

rf_base.fit(X_train_final, y_train_final)

pred_train = rf_base.predict(X_train_final)
pred_val = rf_base.predict(X_val_final)

overfit_table = pd.DataFrame({
    "Métrica": ["RMSE", "MAE", "R²"],
    "Train": [
        np.sqrt(mean_squared_error(y_train_final, pred_train)),
        mean_absolute_error(y_train_final, pred_train),
        r2_score(y_train_final, pred_train)
    ],
    "Validation": [
        np.sqrt(mean_squared_error(y_val_final, pred_val)),
        mean_absolute_error(y_val_final, pred_val),
        r2_score(y_val_final, pred_val)
    ]
})

overfit_table

,Métrica,Train,Validation
0,RMSE,18162.022128,50342.061115
1,MAE,11839.570712,32746.868167
2,R²,0.975242,0.807725


Para verificar la capacidad de generalización del modelo ganador, se comparó su desempeño en entrenamiento y validación. Los resultados muestran un error considerablemente menor en entrenamiento que en validación, lo que indica que el modelo aprende muy bien los patrones del conjunto de entrenamiento, pero también presenta cierto grado de sobreajuste. En tu ejecución, por ejemplo, el RMSE pasó de aproximadamente **18,162** en train a **50,342** en validación, mientras que el R² cayó de **0.975** a **0.808**.

No obstante, este comportamiento sigue siendo sustancialmente mejor que el observado en los otros algoritmos comparados, especialmente frente al árbol de decisión individual. Por ello, el siguiente paso consiste en ajustar los hiperparámetros del bosque para reducir esa brecha sin sacrificar demasiado desempeño.

**FINE-TUNING**

In [40]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_A, y_A)

best_rf = grid_search.best_estimator_

print("Best params:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


/opt/anaconda3/envs/lab4mac/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best CV RMSE: 51464.30510533582


Una vez identificado el mejor modelo base, se realizó una búsqueda de hiperparámetros mediante `GridSearchCV`, enfocada en parámetros que controlan directamente la complejidad del bosque: número de árboles, profundidad máxima, tamaño mínimo de partición, tamaño mínimo de hoja y número de variables consideradas en cada división.

En esta etapa, el objetivo no es solo mejorar el desempeño, sino encontrar una configuración más estable y con mejor capacidad de generalización. Sin embargo, si el modelo ajustado no supera al baseline en validación cruzada, se mantiene como solución final el modelo base ganador, ya que la prioridad metodológica es el desempeño fuera de muestra y no la complejidad por sí misma.

**Nota técnica sobre el proceso de ajuste fino**

Durante el `GridSearchCV` apareció el warning: `A worker stopped while some jobs were given to the executor`

En este caso, el ajuste fino implicó una búsqueda amplia de hiperparámetros con **216 combinaciones y validación cruzada de 5 folds**, lo que generó un total de **1080 ejecuciones** en paralelo. Además, la búsqueda se corrió con `n_jobs=-1`, es decir, utilizando todos los núcleos disponibles del procesador.

Este tipo de warning suele aparecer en procesos paralelos intensivos cuando uno de los workers se detiene o reinicia durante la ejecución. En la práctica, esto suele asociarse a presión de memoria, timeouts o reinicios preventivos de procesos en cargas pesadas de cómputo. Aun así, en este caso el proceso terminó correctamente, devolvió los mejores hiperparámetros y permitió obtener las métricas finales del modelo.

Para una versión más estable, se podría reducir la grilla de búsqueda y utilizar un nivel menor de paralelización (por ejemplo, `n_jobs=2` o `n_jobs=1`) en lugar de `n_jobs=-1`.

**EVALUACION FINAL EN TEST**

In [41]:
final_test_pred = best_rf.predict(X_test_A)

final_rmse = np.sqrt(mean_squared_error(y_test_A, final_test_pred))
final_mae = mean_absolute_error(y_test_A, final_test_pred)
final_r2 = r2_score(y_test_A, final_test_pred)

final_results = pd.DataFrame({
    "Métrica": ["RMSE", "MAE", "R²"],
    "Valor": [final_rmse, final_mae, final_r2]
})

final_results

,Métrica,Valor
0,RMSE,51996.379576
1,MAE,34417.328465
2,R²,0.798025


### Benchmark y Conclusión Final

La experimentación evaluó cuatro algoritmos de regresión (`LinearRegression`, `SGDRegressor`, `DecisionTreeRegressor` y `RandomForestRegressor`) sobre tres configuraciones de variables: set A (originales), set B (con features derivadas base) y set C (con features ampliadas). La comparación mediante validación cruzada mostró que el mejor desempeño corresponde a **RandomForestRegressor con el set A**, alcanzando un RMSE promedio cercano a **49,284** y superando consistentemente al resto de modelos y configuraciones.

Los resultados evidencian que los modelos de ensamble capturan de forma más efectiva las relaciones no lineales del problema, especialmente aquellas asociadas a ingreso, localización y características del entorno. En contraste, los modelos lineales presentaron menor capacidad predictiva, mientras que el árbol de decisión individual mostró mayor sensibilidad al sobreajuste. Asimismo, `SGDRegressor` fue descartado debido a inestabilidad numérica y errores significativamente superiores. Un hallazgo clave es que el feature engineering adicional (sets B y C) no mejoró el desempeño fuera de muestra, lo que indica que aumentar la complejidad del dataset no necesariamente se traduce en una mejor generalización.

Durante el ajuste fino del modelo ganador se exploraron múltiples combinaciones de hiperparámetros mediante `GridSearchCV`. Sin embargo, la mejor configuración obtenida no superó el desempeño del modelo base en validación cruzada. En línea con un criterio de parsimonia y robustez, se opta por mantener como solución final el modelo base ganador. En la evaluación sobre el conjunto de prueba, el modelo alcanzó aproximadamente **RMSE = 51,996**, **MAE = 34,417** y **R² = 0.798**, confirmando una buena capacidad de generalización.

Desde una perspectiva de negocio, estos resultados indican que el valor de la vivienda puede estimarse de manera robusta utilizando variables estructurales ya disponibles, sin necesidad de incrementar artificialmente la complejidad del modelo. El sistema desarrollado permite generar estimaciones consistentes, comparables y escalables, lo que lo convierte en una herramienta útil para análisis exploratorio de precios, identificación de zonas de interés y apoyo en decisiones inmobiliarias. Aunque no reemplaza una tasación profesional, sí mejora significativamente la rapidez y consistencia en la evaluación de oportunidades.

**Modelo final seleccionado:** `RandomForestRegressor` con el set A (variables originales).